In [ ]:
# Step 1: Configuration (replace SEC_USER_AGENT with your institution and email)
import pathlib
import re
import time
from urllib.parse import urljoin
import zipfile

from bs4 import BeautifulSoup, Tag
import numpy as np
import pandas as pd
import requests

SEC_INDEX_URL = "https://www.sec.gov/data-research/sec-markets-data/form-13f-data-sets"
SEC_BASE_URL = "https://www.sec.gov"
SEC_USER_AGENT = "UniversityResearch researcher@university.edu"

START_YEAR = 2013
END_YEAR = 2026

MAX_FILING_LAG_DAYS = 365
TIMELY_FILING_LAG_DAYS = 45
# Because the SEC changed the unit of measure for 13F filings from shares to dollars starting with the 2023Q1 filings
SEC_DOLLAR_UNIT_CUTOFF = pd.Timestamp("2023-01-03")

REQUEST_PAUSE_SECONDS = 0.25
MAX_DOWNLOAD_RETRIES = 3
INFOTABLE_CHUNK_SIZE = 500_000

RAW_ZIP_DIR = pathlib.Path("data/raw_13f_zips")
PROCESSED_DIR = pathlib.Path("data/processed")
OUTPUT_QUARTERLY_IO_FILE = PROCESSED_DIR / "sec_13f_quarterly_io.csv.gz"

RAW_ZIP_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

REQUEST_HEADERS = {
    "User-Agent": SEC_USER_AGENT,
    "Accept-Encoding": "gzip, deflate",
}

In [ ]:
# Step 2: Collect SEC Form 13F ZIP links (include end_year + 1 for Q4 filings submitted in Jan-Feb)
def fetch_sec_13f_zip_urls(
    index_url: str,
    base_url: str,
    headers: dict[str, str],
    start_year: int,
    end_year: int,
) -> list[str]:
    response = requests.get(index_url, headers=headers, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    target_years = set(range(start_year, end_year + 2))
    matched_urls: list[str] = []

    for element in soup.find_all("a", href=True):
        if not isinstance(element, Tag):
            continue
        raw_href = element.get("href")
        if not isinstance(raw_href, str):
            continue
        href_lower = raw_href.strip().lower()
        if not href_lower.endswith(".zip") or "13f" not in href_lower:
            continue

        years_in_name = {int(yr) for yr in re.findall(r"20\d{2}", href_lower)}
        if years_in_name and years_in_name.isdisjoint(target_years):
            continue

        matched_urls.append(urljoin(base_url, raw_href.strip()))

    return list(dict.fromkeys(matched_urls))


zip_urls = fetch_sec_13f_zip_urls(
    index_url=SEC_INDEX_URL,
    base_url=SEC_BASE_URL,
    headers=REQUEST_HEADERS,
    start_year=START_YEAR,
    end_year=END_YEAR,
)
print(f"Discovered {len(zip_urls)} SEC Form 13F ZIP archives for {START_YEAR}-{END_YEAR}.")

In [ ]:
# Step 3: Download SEC 13F ZIP files
def download_zip_archives(
    urls: list[str],
    dest_dir: pathlib.Path,
    headers: dict[str, str],
    pause_seconds: float,
    max_retries: int,
) -> list[pathlib.Path]:
    valid_zip_paths: list[pathlib.Path] = []

    for idx, url in enumerate(urls, start=1):
        file_name = url.rstrip("/").split("/")[-1]
        local_path = dest_dir / file_name

        if local_path.exists() and zipfile.is_zipfile(local_path):
            print(f"[{idx}/{len(urls)}] Cached OK: {file_name}")
            valid_zip_paths.append(local_path)
            continue

        downloaded_ok = False
        for attempt in range(1, max_retries + 1):
            try:
                print(f"[{idx}/{len(urls)}] Downloading {file_name} (attempt {attempt})...")
                with requests.get(url, headers=headers, stream=True, timeout=120) as resp:
                    resp.raise_for_status()
                    with open(local_path, "wb") as out_fh:
                        for chunk in resp.iter_content(chunk_size=65_536):
                            if chunk:
                                out_fh.write(chunk)

                if zipfile.is_zipfile(local_path):
                    downloaded_ok = True
                    valid_zip_paths.append(local_path)
                    break

                local_path.unlink(missing_ok=True)
            except requests.RequestException as exc:
                local_path.unlink(missing_ok=True)
                print(f"  Attempt {attempt} failed for {file_name}: {exc}")

            time.sleep(pause_seconds * (2 ** attempt))

        if not downloaded_ok:
            print(f"  Skipped {file_name} after {max_retries} unsuccessful attempts.")

        time.sleep(pause_seconds)

    return valid_zip_paths


local_zip_files = download_zip_archives(
    urls=zip_urls,
    dest_dir=RAW_ZIP_DIR,
    headers=REQUEST_HEADERS,
    pause_seconds=REQUEST_PAUSE_SECONDS,
    max_retries=MAX_DOWNLOAD_RETRIES,
)
print(f"Ready to parse {len(local_zip_files)} verified ZIP archives.")

In [ ]:
# Step 4: Parse SUBMISSION.tsv + COVERPAGE.tsv and resolve 13F-HR/A amendments
def find_tsv_member(zf: zipfile.ZipFile, filename: str) -> str | None:
    target = filename.upper()
    for member in zf.namelist():
        base_name = member.replace("\\", "/").rsplit("/", 1)[-1].upper()
        if base_name == target:
            return member
    return None


def build_clean_filing_table(
    zip_paths: list[pathlib.Path],
    start_year: int,
    end_year: int,
    max_lag_days: int,
) -> pd.DataFrame:
    filing_frames: list[pd.DataFrame] = []

    for zip_path in sorted(zip_paths):
        with zipfile.ZipFile(zip_path, mode="r") as zf:
            sub_member = find_tsv_member(zf, "SUBMISSION.tsv")
            cov_member = find_tsv_member(zf, "COVERPAGE.tsv")
            if sub_member is None or cov_member is None:
                continue

            with zf.open(sub_member) as sub_fh:
                sub_df = pd.read_csv(
                    sub_fh,
                    sep="\t",
                    dtype=str,
                    usecols=[
                        "ACCESSION_NUMBER",
                        "FILING_DATE",
                        "SUBMISSIONTYPE",
                        "CIK",
                        "PERIODOFREPORT",
                    ],
                )
            with zf.open(cov_member) as cov_fh:
                cov_df = pd.read_csv(
                    cov_fh,
                    sep="\t",
                    dtype=str,
                    usecols=["ACCESSION_NUMBER", "AMENDMENTNO", "AMENDMENTTYPE"],
                )

        cov_df = cov_df.drop_duplicates(subset=["ACCESSION_NUMBER"], keep="last")
        filing_frames.append(pd.merge(sub_df, cov_df, on="ACCESSION_NUMBER", how="left"))

    if not filing_frames:
        raise RuntimeError("No valid SUBMISSION.tsv / COVERPAGE.tsv tables found in RAW_ZIP_DIR.")

    all_filings = pd.concat(filing_frames, ignore_index=True)
    all_filings = all_filings.drop_duplicates(subset=["ACCESSION_NUMBER"], keep="last")

    # Keep holdings reports only (drop 13F-NT notice filings)
    valid_types = all_filings["SUBMISSIONTYPE"].isin(["13F-HR", "13F-HR/A"])
    all_filings = all_filings.loc[valid_types].copy()

    all_filings["rdate"] = pd.to_datetime(
        all_filings["PERIODOFREPORT"], format="%d-%b-%Y", errors="coerce"
    )
    all_filings["fdate"] = pd.to_datetime(
        all_filings["FILING_DATE"], format="%d-%b-%Y", errors="coerce"
    )
    all_filings = all_filings.dropna(subset=["rdate", "fdate", "CIK", "ACCESSION_NUMBER"]).copy()

    # Keep standard calendar quarter-ends within [start_year, end_year]
    is_std_qtr = all_filings["rdate"].dt.is_quarter_end & all_filings["rdate"].dt.month.isin(
        [3, 6, 9, 12]
    )
    in_year_window = all_filings["rdate"].dt.year.between(start_year, end_year)
    filing_lag_days = (all_filings["fdate"] - all_filings["rdate"]).dt.days
    reasonable_lag = filing_lag_days.between(-5, max_lag_days)
    all_filings = all_filings.loc[is_std_qtr & in_year_window & reasonable_lag].copy()

    all_filings["CIK"] = all_filings["CIK"].str.strip().str.zfill(10)
    amend_type_clean = all_filings["AMENDMENTTYPE"].fillna("").str.strip().str.upper()
    all_filings["amend_num"] = pd.to_numeric(
        all_filings["AMENDMENTNO"], errors="coerce"
    ).fillna(0.0)
    all_filings["is_new_holdings"] = (
        all_filings["SUBMISSIONTYPE"] == "13F-HR/A"
    ) & (amend_type_clean == "NEW HOLDINGS")

    # SEC Special Instruction 5: keep latest 13F-HR / RESTATEMENT plus subsequent NEW HOLDINGS
    baseline_pool = all_filings.loc[all_filings["is_new_holdings"].eq(False)].sort_values(
        ["CIK", "rdate", "fdate", "amend_num", "ACCESSION_NUMBER"]
    )
    latest_baseline = baseline_pool.drop_duplicates(subset=["CIK", "rdate"], keep="last").copy()

    baseline_dates = latest_baseline[["CIK", "rdate", "fdate"]].rename(
        columns={"fdate": "baseline_fdate"}
    )
    new_holdings_pool = pd.merge(
        all_filings.loc[all_filings["is_new_holdings"]],
        baseline_dates,
        on=["CIK", "rdate"],
        how="left",
    )
    valid_additions_mask = new_holdings_pool["baseline_fdate"].isna() | (
        new_holdings_pool["fdate"] >= new_holdings_pool["baseline_fdate"]
    )
    valid_additions = new_holdings_pool.loc[valid_additions_mask].drop(columns=["baseline_fdate"])

    keep_cols = [
        "ACCESSION_NUMBER",
        "CIK",
        "rdate",
        "fdate",
        "SUBMISSIONTYPE",
        "is_new_holdings",
    ]
    clean_parts: list[pd.DataFrame] = [
        latest_baseline[keep_cols],
        valid_additions[keep_cols],
    ]
    clean_filings_df = pd.concat(clean_parts, ignore_index=True)
    return clean_filings_df.drop_duplicates(subset=["ACCESSION_NUMBER"])


clean_filings = build_clean_filing_table(
    zip_paths=local_zip_files,
    start_year=START_YEAR,
    end_year=END_YEAR,
    max_lag_days=MAX_FILING_LAG_DAYS,
)
print(
    f"Retained {len(clean_filings):,} clean Form 13F filings across "
    f"{clean_filings['rdate'].nunique()} calendar quarters."
)

In [ ]:
# Step 5: Stream INFOTABLE.tsv and aggregate quarterly equity holdings by (rdate, cusip8)
def aggregate_quarterly_13f_holdings(
    zip_paths: list[pathlib.Path],
    filings_meta: pd.DataFrame,
    dollar_cutoff: pd.Timestamp,
    timely_lag_days: int,
    chunk_size: int,
) -> pd.DataFrame:
    filing_lookup = filings_meta[["ACCESSION_NUMBER", "CIK", "rdate", "fdate"]].copy()
    valid_accession_set = set(filing_lookup["ACCESSION_NUMBER"])
    zip_quarterly_frames: list[pd.DataFrame] = []

    infotable_cols = [
        "ACCESSION_NUMBER",
        "CUSIP",
        "VALUE",
        "SSHPRNAMT",
        "SSHPRNAMTTYPE",
        "PUTCALL",
    ]

    for idx, zip_path in enumerate(sorted(zip_paths), start=1):
        with zipfile.ZipFile(zip_path, mode="r") as zf:
            info_member = find_tsv_member(zf, "INFOTABLE.tsv")
            if info_member is None:
                continue

            manager_chunk_frames: list[pd.DataFrame] = []
            with zf.open(info_member) as info_fh:
                reader = pd.read_csv(
                    info_fh,
                    sep="\t",
                    dtype=str,
                    usecols=infotable_cols,
                    chunksize=chunk_size,
                    low_memory=False,
                )
                for chunk in reader:
                    in_clean = chunk["ACCESSION_NUMBER"].isin(valid_accession_set)
                    sub_chunk = chunk.loc[in_clean].copy()
                    if sub_chunk.empty:
                        continue

                    # Keep share-denominated positions ('SH') and exclude call/put options
                    is_shares = sub_chunk["SSHPRNAMTTYPE"].fillna("").str.strip().str.upper() == "SH"
                    is_non_option = sub_chunk["PUTCALL"].fillna("").str.strip() == ""
                    sub_chunk = sub_chunk.loc[is_shares & is_non_option].copy()
                    if sub_chunk.empty:
                        continue

                    sub_chunk["cusip8"] = sub_chunk["CUSIP"].fillna("").str.strip().str.upper().str[:8]
                    sub_chunk["shares"] = pd.to_numeric(sub_chunk["SSHPRNAMT"], errors="coerce")
                    sub_chunk["raw_value"] = pd.to_numeric(
                        sub_chunk["VALUE"], errors="coerce"
                    ).fillna(0.0)

                    valid_pos = (
                        (sub_chunk["cusip8"].str.len() == 8)
                        & sub_chunk["shares"].notna()
                        & (sub_chunk["shares"] > 0)
                    )
                    sub_chunk = sub_chunk.loc[valid_pos].copy()
                    if sub_chunk.empty:
                        continue

                    sub_chunk = pd.merge(
                        sub_chunk[["ACCESSION_NUMBER", "cusip8", "shares", "raw_value"]],
                        filing_lookup,
                        on="ACCESSION_NUMBER",
                        how="inner",
                    )

                    # Harmonize VALUE to USD ($1,000s before 2023-01-03; $1 on/after; guard against 1,000x unit errors)
                    sub_chunk["val_per_sh"] = sub_chunk["raw_value"] / sub_chunk["shares"]
                    cusip_med_vps = sub_chunk.groupby("cusip8")["val_per_sh"].transform("median")
                    reported_in_dollars_early = (
                        (sub_chunk["fdate"] < dollar_cutoff)
                        & (cusip_med_vps > 0)
                        & (sub_chunk["val_per_sh"] > 250.0 * cusip_med_vps)
                    )
                    unit_multiplier = np.where(
                        reported_in_dollars_early,
                        1.0,
                        np.where(sub_chunk["fdate"] < dollar_cutoff, 1000.0, 1.0),
                    )
                    sub_chunk["value_usd"] = sub_chunk["raw_value"] * unit_multiplier

                    is_timely = (sub_chunk["fdate"] - sub_chunk["rdate"]).dt.days <= timely_lag_days
                    sub_chunk["shares_timely_45d"] = np.where(is_timely, sub_chunk["shares"], 0.0)

                    grouped_chunk = sub_chunk.groupby(
                        ["rdate", "CIK", "cusip8"], as_index=False
                    )[["shares", "shares_timely_45d", "value_usd"]].sum()
                    manager_chunk_frames.append(grouped_chunk)

        if not manager_chunk_frames:
            continue

        zip_manager_df = pd.concat(manager_chunk_frames, ignore_index=True)
        zip_manager_df = zip_manager_df.groupby(
            ["rdate", "CIK", "cusip8"], as_index=False
        )[["shares", "shares_timely_45d", "value_usd"]].sum()
        zip_manager_df["inst_owner_count"] = 1

        zip_stock_df = zip_manager_df.groupby(["rdate", "cusip8"], as_index=False)[
            ["shares", "shares_timely_45d", "value_usd", "inst_owner_count"]
        ].sum()
        zip_quarterly_frames.append(zip_stock_df)
        print(f"[{idx}/{len(zip_paths)}] Aggregated {zip_path.name}: {len(zip_stock_df):,} CUSIP-quarters")

    if not zip_quarterly_frames:
        raise RuntimeError("No valid equity holdings extracted from INFOTABLE.tsv.")

    combined_io = pd.concat(zip_quarterly_frames, ignore_index=True)
    quarterly_io_df = combined_io.groupby(["rdate", "cusip8"], as_index=False)[
        ["shares", "shares_timely_45d", "value_usd", "inst_owner_count"]
    ].sum()
    quarterly_io_df = quarterly_io_df.rename(
        columns={
            "shares": "inst_shares",
            "shares_timely_45d": "inst_shares_timely_45d",
            "value_usd": "inst_value_usd",
            "inst_owner_count": "num_inst_owners",
        }
    )
    return quarterly_io_df.sort_values(["rdate", "cusip8"]).reset_index(drop=True)


quarterly_io = aggregate_quarterly_13f_holdings(
    zip_paths=local_zip_files,
    filings_meta=clean_filings,
    dollar_cutoff=SEC_DOLLAR_UNIT_CUTOFF,
    timely_lag_days=TIMELY_FILING_LAG_DAYS,
    chunk_size=INFOTABLE_CHUNK_SIZE,
)

In [ ]:
# Step 6: Save quarterly institutional ownership and print recent quarters
quarterly_io.to_csv(OUTPUT_QUARTERLY_IO_FILE, index=False, compression="gzip")
print(f"Saved {len(quarterly_io):,} stock-quarter observations to {OUTPUT_QUARTERLY_IO_FILE}")

qtr_summary = quarterly_io.groupby("rdate", as_index=False).agg(
    num_cusips=("cusip8", "count"),
    total_value_trillions=("inst_value_usd", "sum"),
    median_owners=("num_inst_owners", "median"),
)
qtr_summary["total_value_trillions"] = qtr_summary["total_value_trillions"] / 1e12
# Print the last 12 quarters of summary statistics
print(qtr_summary.tail(12).to_string(index=False))